# PeMS 检测站道路路段匹配与可视化

本 notebook 实现：
1. 加载 Caltrans 官方路网数据（SHN_Lines）
2. 根据里程桩（Odometer）匹配 PeMS 检测站对应的道路路段
3. 在地图上可视化检测站及其覆盖的道路路段
4. 支持多种底图切换（OSM、卫星图等）

## 1. 环境配置

In [ ]:
!pip install geopandas shapely folium pandas numpy -q

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from shapely.ops import substring
import folium
import os
import warnings
warnings.filterwarnings('ignore')
print("库加载完成！")

## 2. 配置数据路径

In [ ]:
# Caltrans 路网数据路径
SHN_LINES_PATH = "SHN_Lines/SHN_Lines.shp"

# PeMS 元数据路径
PEMS_META_PATH = "d03_text_meta_2025_01_18.txt"

# 输出目录
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("配置完成！")

## 3. 加载数据

In [ ]:
# 加载 Caltrans 路网数据
print("正在加载 Caltrans 路网数据...")
lines_raw = gpd.read_file(SHN_LINES_PATH)
lines = lines_raw.to_crs(epsg=4326)
print(f"Caltrans 路段数: {len(lines)}")

In [ ]:
# 加载 PeMS 元数据
print("正在加载 PeMS 元数据...")
pems_columns = ['ID', 'Fwy', 'Dir', 'District', 'County', 'City', 'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length', 'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2', 'User_ID_3', 'User_ID_4']
pems = pd.read_csv(PEMS_META_PATH, sep='\t', names=pems_columns, header=0, dtype={'ID': str, 'Fwy': str, 'Dir': str, 'Type': str})

# Length 缺失时置为 0
pems['Length'] = pems['Length'].fillna(0)

print(f"PeMS 检测站数: {len(pems)}")
print(f"Length 为 0 的站点数: {(pems['Length'] == 0).sum()}")
display(pems[['ID', 'Fwy', 'Dir', 'Type', 'Abs_PM', 'Length', 'Name']].head(10))

## 4. 定义匹配函数

In [ ]:
def convert_direction(pems_dir):
    mapping = {'E': 'EB', 'W': 'WB', 'N': 'NB', 'S': 'SB'}
    return mapping.get(pems_dir, pems_dir)

def find_road_segments(station, lines_gdf):
    route = str(station['Fwy'])
    direction = convert_direction(station['Dir'])
    abs_pm = station['Abs_PM']
    length = station['Length']
    
    if pd.isna(abs_pm):
        return None, None
    
    pm_start = abs_pm
    pm_end = abs_pm + length
    
    candidates = lines_gdf[(lines_gdf['Route'].astype(str) == route) & (lines_gdf['Direction'] == direction)].copy()
    
    if len(candidates) == 0:
        candidates = lines_gdf[lines_gdf['Route'].astype(str) == route].copy()
        if len(candidates) == 0:
            return None, None
    
    matched_segments = []
    for _, seg in candidates.iterrows():
        seg_start, seg_end = seg['bOdometer'], seg['eOdometer']
        if seg_start > seg_end:
            seg_start, seg_end = seg_end, seg_start
        
        if length == 0:
            if seg_start <= abs_pm <= seg_end:
                matched_segments.append(seg)
        else:
            if seg_start <= pm_end and seg_end >= pm_start:
                matched_segments.append(seg)
    
    if not matched_segments:
        return None, None
    
    return extract_and_correct(matched_segments, pm_start, pm_end, length)

def extract_and_correct(segments, pm_start, pm_end, length):
    all_coords = []
    
    for seg in segments:
        geom = seg.geometry
        seg_start, seg_end = seg['bOdometer'], seg['eOdometer']
        reversed_pm = False
        if seg_start > seg_end:
            seg_start, seg_end = seg_end, seg_start
            reversed_pm = True
        
        seg_length = seg_end - seg_start
        if seg_length <= 0:
            continue
        
        if length == 0:
            ratio = (pm_start - seg_start) / seg_length
            ratio = max(0, min(1, ratio))
            if reversed_pm:
                ratio = 1 - ratio
            try:
                point = geom.interpolate(ratio, normalized=True)
                return point, (point.y, point.x)
            except:
                continue
        
        ratio_start = max(0, (pm_start - seg_start) / seg_length)
        ratio_end = min(1, (pm_end - seg_start) / seg_length)
        
        if ratio_start >= ratio_end:
            continue
        
        if reversed_pm:
            ratio_start, ratio_end = 1 - ratio_end, 1 - ratio_start
        
        try:
            sub_geom = substring(geom, ratio_start, ratio_end, normalized=True)
            if sub_geom and not sub_geom.is_empty:
                if sub_geom.geom_type == 'LineString':
                    all_coords.extend(list(sub_geom.coords))
                elif sub_geom.geom_type == 'Point':
                    all_coords.append((sub_geom.x, sub_geom.y))
        except:
            continue
    
    if len(all_coords) >= 2:
        line = LineString(all_coords)
        midpoint = line.interpolate(0.5, normalized=True)
        return line, (midpoint.y, midpoint.x)
    elif len(all_coords) == 1:
        point = Point(all_coords[0])
        return point, (point.y, point.x)
    
    return None, None

print("匹配函数定义完成！")

## 5. 执行批量匹配

In [ ]:
def match_all_stations(pems_df, lines_gdf, verbose=True):
    results = []
    matched = 0
    total = len(pems_df)
    
    for idx, station in pems_df.iterrows():
        road_geom, corrected_coords = find_road_segments(station, lines_gdf)
        
        result = {
            'ID': station['ID'], 'Fwy': station['Fwy'], 'Dir': station['Dir'],
            'Type': station['Type'], 'Name': station['Name'], 'Abs_PM': station['Abs_PM'],
            'Length': station['Length'], 'Latitude': station['Latitude'],
            'Longitude': station['Longitude'], 'road_geometry': road_geom,
            'matched': road_geom is not None
        }
        
        if corrected_coords:
            result['corrected_lat'] = corrected_coords[0]
            result['corrected_lon'] = corrected_coords[1]
        else:
            result['corrected_lat'] = station['Latitude']
            result['corrected_lon'] = station['Longitude']
        
        results.append(result)
        if road_geom is not None:
            matched += 1
        
        if verbose and (idx + 1) % 200 == 0:
            print(f"已处理 {idx + 1}/{total}，匹配成功 {matched}")
    
    print(f"\n匹配完成: {matched}/{total} ({matched/total*100:.1f}%)")
    return pd.DataFrame(results)

print("开始匹配...")
matched_df = match_all_stations(pems, lines)

In [ ]:
# 匹配结果统计
print(f"总站点数: {len(matched_df)}")
print(f"匹配成功: {matched_df['matched'].sum()}")
print(f"匹配失败: {(~matched_df['matched']).sum()}")
print(f"\nLength > 0: {(matched_df['Length'] > 0).sum()}")
print(f"Length = 0: {(matched_df['Length'] == 0).sum()}")

## 6. 可视化地图

In [ ]:
def create_road_segment_map(matched_df, title="PeMS Stations", show_original=True, show_corrected=True, show_segments=True, basemap='openstreetmap'):
    """
    basemap 选项:
    - 'openstreetmap': OpenStreetMap
    - 'satellite': Google 卫星图
    - 'esri': ESRI 卫星图
    - 'cartodb': CartoDB 浅色
    - 'dark': CartoDB 深色
    """
    valid_df = matched_df[matched_df['Latitude'].notna() & matched_df['Longitude'].notna()].copy()
    center_lat, center_lon = valid_df['Latitude'].mean(), valid_df['Longitude'].mean()
    
    basemap_options = {
        'openstreetmap': {'tiles': 'OpenStreetMap', 'attr': None},
        'satellite': {'tiles': 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 'attr': 'Google'},
        'esri': {'tiles': 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 'attr': 'ESRI'},
        'cartodb': {'tiles': 'CartoDB positron', 'attr': None},
        'dark': {'tiles': 'CartoDB dark_matter', 'attr': None}
    }
    cfg = basemap_options.get(basemap.lower(), basemap_options['openstreetmap'])
    
    if cfg['attr']:
        m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=cfg['tiles'], attr=cfg['attr'])
    else:
        m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=cfg['tiles'])
    
    type_colors = {'ML': 'blue', 'OR': 'green', 'FR': 'red', 'HV': 'purple', 'FF': 'orange', 'CD': 'darkblue', 'CH': 'gray'}
    
    if show_segments:
        seg_group = folium.FeatureGroup(name='Road Segments')
    if show_original:
        orig_group = folium.FeatureGroup(name='Original Coords')
    if show_corrected:
        corr_group = folium.FeatureGroup(name='Corrected Coords')
    
    for _, row in valid_df.iterrows():
        color = type_colors.get(row['Type'], 'gray')
        popup = f"ID: {row['ID']}<br>Name: {row['Name']}<br>Type: {row['Type']}<br>Length: {row['Length']}"
        
        if show_segments and row['road_geometry'] is not None:
            geom = row['road_geometry']
            if geom.geom_type == 'LineString':
                coords = [(lat, lon) for lon, lat in geom.coords]
                folium.PolyLine(coords, color=color, weight=5, opacity=0.8, popup=popup).add_to(seg_group)
            elif geom.geom_type == 'Point':
                folium.CircleMarker([geom.y, geom.x], radius=6, color=color, fill=True, popup=popup+' (Point)').add_to(seg_group)
        
        if show_original:
            folium.CircleMarker([row['Latitude'], row['Longitude']], radius=4, color='black', fill=True, fillColor=color, fillOpacity=0.7, popup=popup+' (Orig)').add_to(orig_group)
        
        if show_corrected and row['matched'] and pd.notna(row['corrected_lat']):
            folium.CircleMarker([row['corrected_lat'], row['corrected_lon']], radius=4, color=color, fill=True, fillOpacity=1.0, popup=popup+' (Corr)').add_to(corr_group)
    
    if show_segments: seg_group.add_to(m)
    if show_original: orig_group.add_to(m)
    if show_corrected: corr_group.add_to(m)
    folium.LayerControl().add_to(m)
    
    return m

print("可视化函数定义完成！")

In [ ]:
# OpenStreetMap 版本
map_osm = create_road_segment_map(matched_df, title="PeMS D3 - OSM", basemap='openstreetmap')
map_osm.save(os.path.join(OUTPUT_DIR, 'pems_osm.html'))
print("已保存: output/pems_osm.html")
display(map_osm)

In [ ]:
# 卫星图版本
map_sat = create_road_segment_map(matched_df, title="PeMS D3 - Satellite", basemap='satellite')
map_sat.save(os.path.join(OUTPUT_DIR, 'pems_satellite.html'))
print("已保存: output/pems_satellite.html")
display(map_sat)

## 7. 特定高速公路可视化

In [ ]:
def create_freeway_map(matched_df, freeway, direction=None, basemap='satellite'):
    df_fwy = matched_df[matched_df['Fwy'] == str(freeway)].copy()
    if direction:
        df_fwy = df_fwy[df_fwy['Dir'] == direction]
    if len(df_fwy) == 0:
        print(f"未找到 {freeway} {direction or ''} 数据")
        return None
    print(f"筛选到 {len(df_fwy)} 个站点")
    return create_road_segment_map(df_fwy, title=f"I-{freeway} {direction or ''}", basemap=basemap)

# I-80
map_80 = create_freeway_map(matched_df, '80', basemap='satellite')
if map_80:
    map_80.save(os.path.join(OUTPUT_DIR, 'i80_satellite.html'))
    display(map_80)

## 8. 导出数据

In [ ]:
export_df = matched_df.drop(columns=['road_geometry']).copy()
export_df.to_csv(os.path.join(OUTPUT_DIR, 'pems_corrected.csv'), index=False)
print("已保存: output/pems_corrected.csv")
display(export_df[['ID', 'Fwy', 'Dir', 'Type', 'Length', 'Latitude', 'Longitude', 'corrected_lat', 'corrected_lon', 'matched']].head())